# EffTemp-Net SingleFrame — Held-Out Evaluation Only
## RWF-2000 + UCF-Crime Fighting subset + Bus Violence

Notebook ini **hanya melakukan evaluasi inference** menggunakan `best.pt` yang sudah dilatih.
Tidak ada training, optimizer, scheduler, backpropagation, atau model selection.

Model dan preprocessing mengikuti notebook training `EffTempNet_SingleFrame_SpatialOnly_*`:

- backbone: `efficientformerv2_s0`;
- true single-frame / spatial-only;
- cache temporal tetap 16 frame;
- frame yang dipakai model = **index 7 (0-based)** / frame ke-8 dari 16;
- resize internal: 64×64 → 224×224;
- ImageNet normalization;
- classifier identik dengan training;
- checkpoint dimuat dengan `strict=True`;
- prediksi menggunakan softmax + argmax;
- AP dan ROC-AUC memakai probabilitas kelas `Violence`;
- RWF, UCF-Crime Fighting, dan Bus Violence dievaluasi **terpisah** dan tidak dipool.

**Asumsi data:** setiap file video adalah clip dengan clip-level label, dan setiap held-out dataset memiliki kelas NonViolence serta Violence.

In [1]:
# Jalankan sekali pada awal sesi Colab.
!pip -q install timm

In [2]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [3]:
import os
import json
import random
import platform
import warnings
from contextlib import nullcontext
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import timm
from timm import create_model
from tqdm.auto import tqdm

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_recall_fscore_support,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    roc_curve,
    precision_recall_curve,
)

warnings.filterwarnings("ignore", category=UserWarning)

print("Python :", platform.python_version())
print("PyTorch:", torch.__version__)
print("timm   :", timm.__version__)
print("CUDA   :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU    :", torch.cuda.get_device_name(0))

Python : 3.13.15
PyTorch: 2.11.0+cu128
timm   : 1.0.28
CUDA   : True
GPU    : NVIDIA A100-SXM4-40GB


## 1. Konfigurasi

Untuk seed lain, dengan struktur folder training yang sama, **cukup ubah `RUN_SEED`** menjadi `0`, `42`, atau `123`.
`CHECKPOINT_PATH` dan folder output evaluasi berubah otomatis.

Path UCF/Bus di bawah adalah default. Jika folder Google Drive Anda berbeda, ubah `raw_dir` **satu kali**. Setelah cache held-out dibuat, cache yang sama dipakai untuk semua seed.

In [4]:
# ============================================================
# SATU-SATUNYA BARIS YANG DIUBAH ANTAR-SEED
# ============================================================
RUN_SEED = 42   # <-- 0 / 42 / 123

if RUN_SEED not in (0, 42, 123):
    raise ValueError(
        f"RUN_SEED harus 0, 42, atau 123; mendapat {RUN_SEED}."
    )

# ============================================================
# CHECKPOINT
# ============================================================
MODEL_ROOT = Path(
    "/content/drive/MyDrive/IVS/ViolenceDetection/Models/EffTempNet"
)

TRAIN_RUN_ROOT = MODEL_ROOT / (
    f"EffTempNet_SingleFrame_SpatialOnly_NoLS_Seed{RUN_SEED}_v1"
)

CHECKPOINT_PATH = (
    TRAIN_RUN_ROOT
    / "single_frame"
    / f"seed_{RUN_SEED}"
    / "best.pt"
)

# Normalnya None. Isi hanya jika struktur checkpoint Anda berbeda.
CHECKPOINT_PATH_OVERRIDE = None
if CHECKPOINT_PATH_OVERRIDE is not None:
    CHECKPOINT_PATH = Path(CHECKPOINT_PATH_OVERRIDE)

EVAL_OUTPUT_DIR = (
    MODEL_ROOT
    / "HeldOut_Evaluation"
    / "SingleFrame"
    / f"seed_{RUN_SEED}"
)

# ============================================================
# HELD-OUT DATA
# ============================================================
BENCHMARK_ROOT = Path(
    "/content/drive/MyDrive/IVS/ViolenceDataset/Benchmark"
)

# Cache RWF yang sudah digunakan eksperimen sebelumnya.
EXISTING_CACHE_DIR = MODEL_ROOT / "new_saved_dataset"

# Cache UCF/Bus disimpan terpisah dari cache training.
HELDOUT_CACHE_DIR = MODEL_ROOT / "heldout_cache_rgb64_v2"

HELDOUT_SPECS = {
    "RWF": {
        "raw_dir": BENCHMARK_ROOT / "RWF",
        "cache_name": "RWF",
        "cache_dir": EXISTING_CACHE_DIR,
        "nonviolence_dir": None,
        "violence_dir": None,
    },
    "UCF": {
        "raw_dir": BENCHMARK_ROOT / "UCF",
        "cache_name": "UCF_Crime_Fighting",
        "cache_dir": HELDOUT_CACHE_DIR,
        "nonviolence_dir": None,
        "violence_dir": None,
    },
    "BV": {
        "raw_dir": BENCHMARK_ROOT / "BV",
        "cache_name": "BV",
        "cache_dir": HELDOUT_CACHE_DIR,
        "nonviolence_dir": None,
        "violence_dir": None,
    },
}

# Jika nama folder kelas tidak persis canonical, notebook mencoba alias ini.
# Bila ambigu, notebook berhenti dan meminta mapping eksplisit.
NONVIOLENCE_ALIASES = (
    "NonViolence", "Non-Violence", "Non_Violence",
    "Nonviolent", "NonViolent",
    "NonFight", "Non-Fight", "Non_Fight",
    "Normal", "NormalVideos", "Normal_Videos",
)

VIOLENCE_ALIASES = (
    "Violence", "Violent",
    "Fight", "Fighting", "FightVideos",
    "ViolentVideos", "Abnormal",
)

# ============================================================
# PREPROCESSING — HARUS TETAP SAMA DENGAN TRAINING
# ============================================================
CLASS_NAMES = ["NonViolence", "Violence"]
NUM_CLASSES = 2

SEQUENCE_LENGTH = 16
MODEL_INPUT_FRAMES = 1
SINGLE_FRAME_INDEX = (SEQUENCE_LENGTH - 1) // 2  # 7
SINGLE_FRAME_NUMBER_1BASED = SINGLE_FRAME_INDEX + 1

RAW_HEIGHT = 64
RAW_WIDTH = 64
BACKBONE_INPUT_SIZE = 224

AUTO_SCALE_LEGACY_CACHE = True
LEGACY_CACHE_COLOR_ORDER = "BGR"

USE_AMP = False
EVAL_BATCH_SIZE = 16
NUM_WORKERS = 2
BOOTSTRAP_SAMPLES = 1000
BOOTSTRAP_SEED = 2026
ALLOW_BUILD_CACHE = True

EVAL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
HELDOUT_CACHE_DIR.mkdir(parents=True, exist_ok=True)

print("RUN_SEED        :", RUN_SEED)
print("CHECKPOINT_PATH :", CHECKPOINT_PATH)
print("OUTPUT          :", EVAL_OUTPUT_DIR)
print("Frame index     :", SINGLE_FRAME_INDEX)

RUN_SEED        : 42
CHECKPOINT_PATH : /content/drive/MyDrive/IVS/ViolenceDetection/Models/EffTempNet/EffTempNet_SingleFrame_SpatialOnly_NoLS_Seed42_v1/single_frame/seed_42/best.pt
OUTPUT          : /content/drive/MyDrive/IVS/ViolenceDetection/Models/EffTempNet/HeldOut_Evaluation/SingleFrame/seed_42
Frame index     : 7


In [5]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
AMP_ENABLED = bool(USE_AMP and DEVICE.type == "cuda")
AMP_DTYPE = (
    torch.bfloat16
    if AMP_ENABLED and torch.cuda.is_bf16_supported()
    else torch.float16
)

def amp_context():
    if not AMP_ENABLED:
        return nullcontext()
    return torch.autocast(
        device_type="cuda",
        dtype=AMP_DTYPE,
        enabled=True,
    )

def set_global_seed(seed: int) -> None:
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def seed_worker(worker_id: int) -> None:
    worker_seed = torch.initial_seed() % (2**32)
    np.random.seed(worker_seed)
    random.seed(worker_seed)

def assert_finite_tensor(name: str, tensor: torch.Tensor) -> None:
    if not torch.isfinite(tensor).all():
        nan_count = int(torch.isnan(tensor).sum().item())
        inf_count = int(torch.isinf(tensor).sum().item())
        raise FloatingPointError(
            f"{name} contains NaN/Inf: nan={nan_count}, inf={inf_count}"
        )

def safe_torch_load(path: Path, map_location):
    try:
        return torch.load(
            path,
            map_location=map_location,
            weights_only=False,
        )
    except TypeError:
        return torch.load(path, map_location=map_location)

set_global_seed(BOOTSTRAP_SEED)
print("Device:", DEVICE, "| AMP:", AMP_ENABLED)

Device: cuda | AMP: False


## 2. Model SingleFrame

Definisi model disalin dari arsitektur training. Saat evaluasi dipakai `pretrained=False` karena seluruh bobot dimuat dari `best.pt`.

In [6]:
class EfficientFormerSingleFrame(nn.Module):
    VALID_VARIANTS = {"single_frame"}

    def __init__(
        self,
        variant: str = "single_frame",
        backbone_name: str = "efficientformerv2_s0",
        num_classes: int = NUM_CLASSES,
        pretrained: bool = False,
    ):
        super().__init__()

        if variant not in self.VALID_VARIANTS:
            raise ValueError(f"Unknown variant: {variant}")

        self.variant = variant

        self.backbone = create_model(
            backbone_name,
            pretrained=pretrained,
            num_classes=0,
            in_chans=3,
        )
        self.feature_dim = int(self.backbone.num_features)

        self.feature_dropout = nn.Dropout(0.2)

        self.classifier = nn.Sequential(
            nn.Linear(self.feature_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.25),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.25),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(32, num_classes),
        )

        self.lstm = None
        self.attention = None
        self.temporal_pool = None
        self.tsm = None

    def forward(self, clip: torch.Tensor, return_attention: bool = False):
        if clip.ndim != 5:
            raise ValueError(
                f"Expected [B,1,C,H,W], got {tuple(clip.shape)}"
            )

        batch_size, seq_len, channels, height, width = clip.shape

        if seq_len != MODEL_INPUT_FRAMES:
            raise ValueError(
                f"SingleFrame expects {MODEL_INPUT_FRAMES} frame, got {seq_len}."
            )
        if channels != 3:
            raise ValueError(f"Expected 3 channels, got {channels}.")

        x = clip[:, 0]

        x = F.interpolate(
            x,
            size=(BACKBONE_INPUT_SIZE, BACKBONE_INPUT_SIZE),
            mode="bilinear",
            align_corners=False,
        )

        features = self.backbone(x)

        if features.ndim == 4:
            features = F.adaptive_avg_pool2d(
                features,
                output_size=1,
            ).flatten(1)
        elif features.ndim != 2:
            features = features.flatten(1)

        features = self.feature_dropout(features)
        logits = self.classifier(features)

        if return_attention:
            return logits, None
        return logits

## 3. Cache held-out

Jika cache belum ada, raw video diekstrak dengan prosedur training: 16 posisi uniform, BGR→RGB, 64×64, skala [0,1]. Cache dibuat sekali dan dipakai ulang oleh semua seed.

In [7]:
VIDEO_EXTENSIONS = {
    ".mp4", ".avi", ".mov", ".mkv",
    ".mpeg", ".mpg", ".webm",
}

def direct_subdirs(root: Path) -> Dict[str, Path]:
    if not root.exists():
        return {}
    return {p.name: p for p in root.iterdir() if p.is_dir()}

def resolve_one_class_dir(
    root: Path,
    explicit_name: Optional[str],
    aliases: Tuple[str, ...],
    canonical_name: str,
) -> Path:
    if explicit_name is not None:
        candidate = root / explicit_name
        if not candidate.exists():
            raise FileNotFoundError(
                f"Folder {canonical_name} tidak ditemukan: {candidate}"
            )
        return candidate

    subdirs = direct_subdirs(root)
    lookup = {name.lower(): path for name, path in subdirs.items()}

    matches = []
    for alias in aliases:
        candidate = lookup.get(alias.lower())
        if candidate is not None and candidate not in matches:
            matches.append(candidate)

    if len(matches) != 1:
        raise RuntimeError(
            f"Tidak dapat menentukan folder kelas {canonical_name} pada {root}.\n"
            f"Matches={ [p.name for p in matches] }\n"
            f"Subfolders={sorted(subdirs.keys())}\n"
            "Isi nonviolence_dir/violence_dir secara eksplisit di HELDOUT_SPECS."
        )

    return matches[0]

def list_videos_for_spec(
    raw_dir: Path,
    nonviolence_dir: Optional[str],
    violence_dir: Optional[str],
) -> List[Tuple[Path, int]]:
    if not raw_dir.exists():
        raise FileNotFoundError(
            f"Raw held-out directory tidak ditemukan: {raw_dir}"
        )

    class_dirs = {
        0: resolve_one_class_dir(
            raw_dir, nonviolence_dir,
            NONVIOLENCE_ALIASES, "NonViolence"
        ),
        1: resolve_one_class_dir(
            raw_dir, violence_dir,
            VIOLENCE_ALIASES, "Violence"
        ),
    }

    records = []

    for label, class_dir in class_dirs.items():
        files = sorted(
            p for p in class_dir.rglob("*")
            if p.is_file() and p.suffix.lower() in VIDEO_EXTENSIONS
        )

        if not files:
            raise RuntimeError(
                f"Tidak ada video pada folder kelas: {class_dir}"
            )

        records.extend((p, label) for p in files)

        print(
            f"{CLASS_NAMES[label]:12s}: {len(files):5d} | {class_dir}"
        )

    labels = np.asarray([label for _, label in records], dtype=np.int64)

    if set(np.unique(labels).tolist()) != {0, 1}:
        raise RuntimeError(
            "Held-out dataset harus memiliki kedua kelas 0 dan 1."
        )

    return records

def extract_uniform_rgb_frames(
    video_path: Path,
    sequence_length: int = SEQUENCE_LENGTH,
    height: int = RAW_HEIGHT,
    width: int = RAW_WIDTH,
) -> Optional[np.ndarray]:
    cap = cv2.VideoCapture(str(video_path))

    if not cap.isOpened():
        return None

    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    if frame_count <= 0:
        cap.release()
        return None

    frame_positions = np.linspace(
        0,
        max(frame_count - 1, 0),
        sequence_length,
    ).astype(int)

    frames = []

    for pos in frame_positions:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(pos))
        ok, frame_bgr = cap.read()

        if not ok:
            break

        frame_rgb = cv2.cvtColor(
            frame_bgr,
            cv2.COLOR_BGR2RGB,
        )
        frame_rgb = cv2.resize(
            frame_rgb,
            (width, height),
            interpolation=cv2.INTER_LINEAR,
        )
        frames.append(
            frame_rgb.astype(np.float32) / 255.0
        )

    cap.release()

    if not frames:
        return None

    while len(frames) < sequence_length:
        frames.append(frames[-1].copy())

    return np.stack(
        frames[:sequence_length],
        axis=0,
    ).astype(np.float32)


def build_rgb_cache(
    raw_dir: Path,
    cache_dir: Path,
    cache_name: str,
    nonviolence_dir: Optional[str],
    violence_dir: Optional[str],
) -> Tuple[Path, Path, Path, str]:
    cache_dir.mkdir(parents=True, exist_ok=True)

    feature_path = cache_dir / f"{cache_name}_rgb64_v2_features.npy"
    label_path = cache_dir / f"{cache_name}_rgb64_v2_labels.npy"
    paths_path = cache_dir / f"{cache_name}_rgb64_v2_paths.json"

    records = list_videos_for_spec(
        raw_dir,
        nonviolence_dir,
        violence_dir,
    )

    n_records = len(records)
    if n_records == 0:
        raise RuntimeError(f"Tidak ada video untuk {cache_name}.")

    # Temporary memmap: tidak menumpuk seluruh dataset ke RAM.
    tmp_feature_path = (
        cache_dir / f".{cache_name}_rgb64_v2_features.tmp.npy"
    )

    if tmp_feature_path.exists():
        tmp_feature_path.unlink()

    tmp_features = np.lib.format.open_memmap(
        tmp_feature_path,
        mode="w+",
        dtype=np.float32,
        shape=(
            n_records,
            SEQUENCE_LENGTH,
            RAW_HEIGHT,
            RAW_WIDTH,
            3,
        ),
    )

    labels_buffer = np.empty(
        n_records,
        dtype=np.int64,
    )

    valid_paths = []
    failed_paths = []
    valid_count = 0

    for video_path, label in tqdm(
        records,
        desc=f"Build cache {cache_name}",
        unit="video",
    ):
        clip = extract_uniform_rgb_frames(video_path)

        if clip is None:
            failed_paths.append(str(video_path))
            continue

        tmp_features[valid_count] = clip
        labels_buffer[valid_count] = int(label)
        valid_paths.append(str(video_path))
        valid_count += 1

    tmp_features.flush()
    del tmp_features

    if valid_count == 0:
        if tmp_feature_path.exists():
            tmp_feature_path.unlink()
        raise RuntimeError(
            f"Semua video gagal didekode untuk {cache_name}."
        )

    labels_valid = labels_buffer[:valid_count]

    if set(np.unique(labels_valid).tolist()) != {0, 1}:
        if tmp_feature_path.exists():
            tmp_feature_path.unlink()
        raise RuntimeError(
            f"Setelah decode, {cache_name} tidak memiliki kedua kelas."
        )

    # Trim temporary cache ke jumlah clip valid secara chunked.
    tmp_read = np.load(
        tmp_feature_path,
        mmap_mode="r",
    )

    final_features = np.lib.format.open_memmap(
        feature_path,
        mode="w+",
        dtype=np.float32,
        shape=(
            valid_count,
            SEQUENCE_LENGTH,
            RAW_HEIGHT,
            RAW_WIDTH,
            3,
        ),
    )

    chunk_size = 32

    for begin in range(0, valid_count, chunk_size):
        finish = min(begin + chunk_size, valid_count)
        final_features[begin:finish] = tmp_read[begin:finish]

    final_features.flush()
    del final_features, tmp_read

    np.save(
        label_path,
        labels_valid,
    )

    with open(paths_path, "w", encoding="utf-8") as f:
        json.dump(
            {
                "paths": valid_paths,
                "failed": failed_paths,
                "n_requested": n_records,
                "n_valid": valid_count,
                "color_order": "RGB",
                "sequence_length": SEQUENCE_LENGTH,
                "height": RAW_HEIGHT,
                "width": RAW_WIDTH,
            },
            f,
            indent=2,
        )

    if tmp_feature_path.exists():
        tmp_feature_path.unlink()

    print(
        "Saved:",
        feature_path,
        "| shape=",
        (
            valid_count,
            SEQUENCE_LENGTH,
            RAW_HEIGHT,
            RAW_WIDTH,
            3,
        ),
    )
    print("Failed videos:", len(failed_paths))

    return feature_path, label_path, paths_path, "RGB"

def resolve_heldout_store(dataset_name: str, spec: Dict) -> Dict:
    raw_dir = Path(spec["raw_dir"])
    cache_dir = Path(spec["cache_dir"])
    cache_name = str(spec["cache_name"])

    v2_feature = cache_dir / f"{cache_name}_rgb64_v2_features.npy"
    v2_label = cache_dir / f"{cache_name}_rgb64_v2_labels.npy"
    v2_paths = cache_dir / f"{cache_name}_rgb64_v2_paths.json"

    legacy_feature = cache_dir / f"{cache_name}_features.npy"
    legacy_label = cache_dir / f"{cache_name}_labels.npy"
    legacy_paths = cache_dir / f"{cache_name}_paths.json"

    if v2_feature.exists() and v2_label.exists():
        print(f"✅ {dataset_name}: RGB-v2 cache")
        return {
            "feature_path": v2_feature,
            "label_path": v2_label,
            "paths_path": v2_paths if v2_paths.exists() else None,
            "color_order": "RGB",
            "cache_source": "rgb64_v2",
        }

    if legacy_feature.exists() and legacy_label.exists():
        print(f"✅ {dataset_name}: legacy cache, BGR→RGB")
        return {
            "feature_path": legacy_feature,
            "label_path": legacy_label,
            "paths_path": legacy_paths if legacy_paths.exists() else None,
            "color_order": LEGACY_CACHE_COLOR_ORDER,
            "cache_source": "legacy",
        }

    if not ALLOW_BUILD_CACHE:
        raise FileNotFoundError(
            f"Cache tidak ada untuk {dataset_name}; ALLOW_BUILD_CACHE=False."
        )

    print(f"ℹ️ {dataset_name}: build cache dari {raw_dir}")

    feature_path, label_path, paths_path, color_order = build_rgb_cache(
        raw_dir=raw_dir,
        cache_dir=cache_dir,
        cache_name=cache_name,
        nonviolence_dir=spec.get("nonviolence_dir"),
        violence_dir=spec.get("violence_dir"),
    )

    return {
        "feature_path": feature_path,
        "label_path": label_path,
        "paths_path": paths_path,
        "color_order": color_order,
        "cache_source": "built_rgb64_v2",
    }

In [8]:
IMAGENET_MEAN = torch.tensor(
    [0.485, 0.456, 0.406],
    dtype=torch.float32,
).view(1, 3, 1, 1)

IMAGENET_STD = torch.tensor(
    [0.229, 0.224, 0.225],
    dtype=torch.float32,
).view(1, 3, 1, 1)

class HeldOutSingleFrameDataset(Dataset):
    def __init__(
        self,
        feature_path: Path,
        label_path: Path,
        color_order: str,
    ):
        self.feature_path = Path(feature_path)
        self.label_path = Path(label_path)
        self.color_order = str(color_order).upper()

        self.features = np.load(self.feature_path, mmap_mode="r")
        self.labels = np.load(
            self.label_path,
            mmap_mode="r",
        ).astype(np.int64)

        if len(self.features) != len(self.labels):
            raise ValueError(
                f"Feature/label mismatch: {len(self.features)} vs {len(self.labels)}"
            )

        expected_tail = (
            SEQUENCE_LENGTH,
            RAW_HEIGHT,
            RAW_WIDTH,
            3,
        )

        if self.features.ndim != 5 or tuple(self.features.shape[1:]) != expected_tail:
            raise ValueError(
                f"Unexpected cache shape {self.features.shape}; "
                f"expected [N,{SEQUENCE_LENGTH},{RAW_HEIGHT},{RAW_WIDTH},3]."
            )

        unique_labels = set(np.unique(self.labels).tolist())
        if unique_labels != {0, 1}:
            raise ValueError(
                f"Held-out labels harus {{0,1}}, ditemukan {sorted(unique_labels)}."
            )

        if self.color_order not in {"RGB", "BGR"}:
            raise ValueError(
                f"Unsupported color_order={self.color_order}"
            )

    def __len__(self) -> int:
        return len(self.labels)

    def __getitem__(self, index: int):
        frames = np.asarray(
            self.features[index],
            dtype=np.float32,
        )
        label = int(self.labels[index])

        if not np.isfinite(frames).all():
            raise ValueError(
                f"NaN/Inf pada sample {index}: {self.feature_path}"
            )

        frame_min = float(frames.min())
        frame_max = float(frames.max())

        if (
            AUTO_SCALE_LEGACY_CACHE
            and frame_min >= 0.0
            and frame_max > 1.5
        ):
            if frame_max <= 255.0 + 1e-3:
                frames = frames / 255.0
            else:
                raise ValueError(
                    f"Invalid range [{frame_min},{frame_max}] sample={index}"
                )

        if float(frames.min()) < -1e-4 or float(frames.max()) > 1.0001:
            raise ValueError(
                f"Cache harus [0,1], mendapat "
                f"[{float(frames.min())},{float(frames.max())}]"
            )

        if self.color_order == "BGR":
            frames = frames[..., [2, 1, 0]]

        selected = frames[
            SINGLE_FRAME_INDEX:SINGLE_FRAME_INDEX + 1
        ]

        if selected.shape != (1, RAW_HEIGHT, RAW_WIDTH, 3):
            raise ValueError(
                f"Selected frame shape {selected.shape} sample={index}"
            )

        clip = torch.from_numpy(
            np.ascontiguousarray(selected)
        ).permute(0, 3, 1, 2)

        clip = (clip - IMAGENET_MEAN) / IMAGENET_STD

        return (
            clip,
            torch.tensor(label, dtype=torch.long),
            int(index),
        )

def make_loader(dataset: Dataset) -> DataLoader:
    generator = torch.Generator()
    generator.manual_seed(BOOTSTRAP_SEED)

    return DataLoader(
        dataset,
        batch_size=EVAL_BATCH_SIZE,
        shuffle=False,
        drop_last=False,
        num_workers=NUM_WORKERS,
        pin_memory=(DEVICE.type == "cuda"),
        worker_init_fn=seed_worker,
        generator=generator,
        persistent_workers=(NUM_WORKERS > 0),
    )

def load_paths_if_available(
    paths_path: Optional[Path],
    expected_n: int,
) -> Optional[List[str]]:
    if paths_path is None:
        return None

    paths_path = Path(paths_path)
    if not paths_path.exists():
        return None

    with open(paths_path, "r", encoding="utf-8") as f:
        payload = json.load(f)

    paths = payload.get("paths")
    if not isinstance(paths, list) or len(paths) != expected_n:
        print(
            f"⚠️ paths.json tidak cocok dengan N={expected_n}; video_path tidak disimpan."
        )
        return None

    return [str(p) for p in paths]

def audit_dataset(dataset_name: str, store: Dict) -> Dict:
    ds = HeldOutSingleFrameDataset(
        store["feature_path"],
        store["label_path"],
        store["color_order"],
    )

    labels = np.asarray(ds.labels, dtype=np.int64)
    counts = np.bincount(labels, minlength=2)

    audit_indices = np.unique(
        np.linspace(
            0,
            len(ds) - 1,
            min(8, len(ds)),
        ).astype(int)
    )

    for idx in audit_indices:
        clip, label, sample_idx = ds[int(idx)]
        if clip.shape != (
            MODEL_INPUT_FRAMES,
            3,
            RAW_HEIGHT,
            RAW_WIDTH,
        ):
            raise RuntimeError(
                f"{dataset_name}: input shape salah {clip.shape}"
            )
        assert_finite_tensor(
            f"{dataset_name}_sample_{sample_idx}",
            clip,
        )

    print(
        f"✅ {dataset_name}: N={len(ds)}, "
        f"NonViolence={counts[0]}, Violence={counts[1]}, "
        f"cache={store['cache_source']}"
    )

    return {
        "dataset": dataset_name,
        "n_total": int(len(ds)),
        "n_nonviolence": int(counts[0]),
        "n_violence": int(counts[1]),
        "feature_path": str(store["feature_path"]),
        "label_path": str(store["label_path"]),
        "color_order_in_cache": store["color_order"],
        "cache_source": store["cache_source"],
        "single_frame_index_0based": SINGLE_FRAME_INDEX,
        "single_frame_number_1based": SINGLE_FRAME_NUMBER_1BASED,
    }

## 4. Load dan audit `best.pt`

Evaluasi **wajib menggunakan `best.pt`**, bukan `last.pt`. Metadata checkpoint dan arsitektur diverifikasi sebelum inference.

In [9]:
if not CHECKPOINT_PATH.exists():
    raise FileNotFoundError(
        "best.pt tidak ditemukan.\n"
        f"Expected: {CHECKPOINT_PATH}\n"
        "Jika struktur folder berbeda, isi CHECKPOINT_PATH_OVERRIDE."
    )

checkpoint = safe_torch_load(
    CHECKPOINT_PATH,
    map_location="cpu",
)

if not isinstance(checkpoint, dict):
    raise TypeError(
        "Checkpoint bukan dictionary dari notebook training."
    )

if "model_state_dict" not in checkpoint:
    raise KeyError(
        "Checkpoint tidak memiliki model_state_dict."
    )

checkpoint_config = checkpoint.get("config", {})

expected_config = {
    "variant": "single_frame",
    "seed": RUN_SEED,
    "cache_sequence_length": SEQUENCE_LENGTH,
    "model_input_frames": MODEL_INPUT_FRAMES,
    "single_frame_index_0based": SINGLE_FRAME_INDEX,
}

mismatches = []

for key, expected in expected_config.items():
    if key in checkpoint_config:
        actual = checkpoint_config[key]
        if actual != expected:
            mismatches.append((key, actual, expected))

if mismatches:
    detail = "\n".join(
        f"- {key}: actual={actual}, expected={expected}"
        for key, actual, expected in mismatches
    )
    raise RuntimeError(
        "Checkpoint configuration mismatch:\n" + detail
    )

model = EfficientFormerSingleFrame(
    variant="single_frame",
    pretrained=False,
).to(DEVICE)

load_result = model.load_state_dict(
    checkpoint["model_state_dict"],
    strict=True,
)

model.eval()

assert model.lstm is None
assert model.attention is None
assert model.temporal_pool is None
assert model.tsm is None

with torch.inference_mode():
    dummy = torch.zeros(
        2,
        MODEL_INPUT_FRAMES,
        3,
        RAW_HEIGHT,
        RAW_WIDTH,
        device=DEVICE,
    )
    dummy_logits = model(dummy)

if tuple(dummy_logits.shape) != (2, NUM_CLASSES):
    raise RuntimeError(
        f"Unexpected output shape {tuple(dummy_logits.shape)}"
    )

print("✅ Checkpoint audit PASS")
print("Checkpoint :", CHECKPOINT_PATH)
print("Epoch      :", checkpoint.get("epoch", "unknown"))
print("Best val F1:", checkpoint.get("best_score", "unknown"))
print("Load result:", load_result)

del dummy, dummy_logits
if torch.cuda.is_available():
    torch.cuda.empty_cache()

✅ Checkpoint audit PASS
Checkpoint : /content/drive/MyDrive/IVS/ViolenceDetection/Models/EffTempNet/EffTempNet_SingleFrame_SpatialOnly_NoLS_Seed42_v1/single_frame/seed_42/best.pt
Epoch      : 42
Best val F1: 0.8197486907164326
Load result: <All keys matched successfully>


## 5. Resolve dan audit RWF, UCF-Crime Fighting, dan Bus Violence

In [10]:
heldout_stores = {}
audit_rows = []

for dataset_name, spec in HELDOUT_SPECS.items():
    print("\n" + "=" * 90)
    print("DATASET:", dataset_name)
    print("=" * 90)

    store = resolve_heldout_store(
        dataset_name,
        spec,
    )

    heldout_stores[dataset_name] = store
    audit_rows.append(
        audit_dataset(dataset_name, store)
    )

dataset_audit_df = pd.DataFrame(audit_rows)

dataset_audit_path = (
    EVAL_OUTPUT_DIR / "heldout_dataset_audit.csv"
)
dataset_audit_df.to_csv(
    dataset_audit_path,
    index=False,
)

display(dataset_audit_df)
print("Saved:", dataset_audit_path)


DATASET: RWF
✅ RWF: legacy cache, BGR→RGB
✅ RWF: N=1998, NonViolence=1000, Violence=998, cache=legacy

DATASET: UCF
ℹ️ UCF: build cache dari /content/drive/MyDrive/IVS/ViolenceDataset/Benchmark/UCF
NonViolence :    50 | /content/drive/MyDrive/IVS/ViolenceDataset/Benchmark/UCF/NonViolence
Violence    :    50 | /content/drive/MyDrive/IVS/ViolenceDataset/Benchmark/UCF/Violence


Build cache UCF_Crime_Fighting:   0%|          | 0/100 [00:00<?, ?video/s]

Saved: /content/drive/MyDrive/IVS/ViolenceDetection/Models/EffTempNet/heldout_cache_rgb64_v2/UCF_Crime_Fighting_rgb64_v2_features.npy | shape= (100, 16, 64, 64, 3)
Failed videos: 0
✅ UCF: N=100, NonViolence=50, Violence=50, cache=built_rgb64_v2

DATASET: BV
ℹ️ BV: build cache dari /content/drive/MyDrive/IVS/ViolenceDataset/Benchmark/BV
NonViolence :   700 | /content/drive/MyDrive/IVS/ViolenceDataset/Benchmark/BV/NonViolence
Violence    :   700 | /content/drive/MyDrive/IVS/ViolenceDataset/Benchmark/BV/Violence


Build cache BV:   0%|          | 0/1400 [00:00<?, ?video/s]

Saved: /content/drive/MyDrive/IVS/ViolenceDetection/Models/EffTempNet/heldout_cache_rgb64_v2/BV_rgb64_v2_features.npy | shape= (1400, 16, 64, 64, 3)
Failed videos: 0
✅ BV: N=1400, NonViolence=700, Violence=700, cache=built_rgb64_v2


,dataset,n_total,n_nonviolence,n_violence,feature_path,label_path,color_order_in_cache,cache_source,single_frame_index_0based,single_frame_number_1based
0,RWF,1998,1000,998,/content/drive/MyDrive/IVS/ViolenceDetection/M...,/content/drive/MyDrive/IVS/ViolenceDetection/M...,BGR,legacy,7,8
1,UCF,100,50,50,/content/drive/MyDrive/IVS/ViolenceDetection/M...,/content/drive/MyDrive/IVS/ViolenceDetection/M...,RGB,built_rgb64_v2,7,8
2,BV,1400,700,700,/content/drive/MyDrive/IVS/ViolenceDetection/M...,/content/drive/MyDrive/IVS/ViolenceDetection/M...,RGB,built_rgb64_v2,7,8


Saved: /content/drive/MyDrive/IVS/ViolenceDetection/Models/EffTempNet/HeldOut_Evaluation/SingleFrame/seed_42/heldout_dataset_audit.csv


## 6. Metrik evaluasi

AP adalah metrik ranking utama; ROC-AUC dan Macro-F1 disimpan sebagai pendamping. Ketiga domain tidak dipool.

In [11]:
def compute_classification_metrics(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    p_violence: np.ndarray,
) -> Dict[str, float]:
    precision, recall, f1, support = precision_recall_fscore_support(
        y_true,
        y_pred,
        labels=[0, 1],
        zero_division=0,
    )

    result = {
        "n": int(len(y_true)),
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "balanced_accuracy": float(
            balanced_accuracy_score(y_true, y_pred)
        ),
        "precision_nonviolence": float(precision[0]),
        "recall_nonviolence": float(recall[0]),
        "f1_nonviolence": float(f1[0]),
        "precision_violence": float(precision[1]),
        "recall_violence": float(recall[1]),
        "f1_violence": float(f1[1]),
        "f1_macro": float(
            f1_score(
                y_true,
                y_pred,
                average="macro",
                zero_division=0,
            )
        ),
    }

    if len(np.unique(y_true)) == 2:
        result["roc_auc"] = float(
            roc_auc_score(y_true, p_violence)
        )
        result["average_precision"] = float(
            average_precision_score(y_true, p_violence)
        )
    else:
        result["roc_auc"] = float("nan")
        result["average_precision"] = float("nan")

    return result

@torch.inference_mode()
def evaluate_loader(
    model: nn.Module,
    loader: DataLoader,
    device: torch.device,
):
    model.eval()

    all_indices = []
    all_labels = []
    all_predictions = []
    all_probabilities = []

    for clips, labels, indices in tqdm(
        loader,
        desc="Inference",
        leave=False,
    ):
        clips = clips.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        assert_finite_tensor("evaluation_input", clips)

        with amp_context():
            logits = model(clips)

        assert_finite_tensor("evaluation_logits", logits)

        probabilities = torch.softmax(
            logits.float(),
            dim=1,
        )

        assert_finite_tensor(
            "evaluation_probabilities",
            probabilities,
        )

        predictions = torch.argmax(
            probabilities,
            dim=1,
        )

        all_indices.extend(
            indices.cpu().numpy().tolist()
        )
        all_labels.extend(
            labels.cpu().numpy().tolist()
        )
        all_predictions.extend(
            predictions.cpu().numpy().tolist()
        )
        all_probabilities.extend(
            probabilities.cpu().numpy().tolist()
        )

    y_true = np.asarray(all_labels, dtype=np.int64)
    y_pred = np.asarray(all_predictions, dtype=np.int64)
    probs = np.asarray(all_probabilities, dtype=np.float64)
    sample_indices = np.asarray(all_indices, dtype=np.int64)

    if len(y_true) == 0:
        raise RuntimeError("Evaluation menghasilkan 0 sample.")

    if probs.shape != (len(y_true), NUM_CLASSES):
        raise RuntimeError(
            f"Invalid probability shape {probs.shape}"
        )

    metrics = compute_classification_metrics(
        y_true,
        y_pred,
        probs[:, 1],
    )

    return metrics, sample_indices, y_true, y_pred, probs

def bootstrap_ci(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    p_violence: np.ndarray,
    metric_name: str,
    n_bootstrap: int,
    seed: int,
    alpha: float = 0.05,
) -> Tuple[float, float]:
    rng = np.random.default_rng(seed)
    n = len(y_true)
    values = []
    attempts = 0
    max_attempts = max(n_bootstrap * 5, 1000)

    while len(values) < n_bootstrap and attempts < max_attempts:
        attempts += 1
        idx = rng.integers(0, n, size=n)

        yt = y_true[idx]
        yp = y_pred[idx]
        pp = p_violence[idx]

        if (
            metric_name in {"roc_auc", "average_precision"}
            and len(np.unique(yt)) < 2
        ):
            continue

        if metric_name == "accuracy":
            value = accuracy_score(yt, yp)
        elif metric_name == "f1_violence":
            value = f1_score(
                yt,
                yp,
                pos_label=1,
                zero_division=0,
            )
        elif metric_name == "f1_macro":
            value = f1_score(
                yt,
                yp,
                average="macro",
                zero_division=0,
            )
        elif metric_name == "roc_auc":
            value = roc_auc_score(yt, pp)
        elif metric_name == "average_precision":
            value = average_precision_score(yt, pp)
        else:
            raise ValueError(metric_name)

        values.append(float(value))

    if not values:
        return float("nan"), float("nan")

    return (
        float(np.quantile(values, alpha / 2)),
        float(np.quantile(values, 1 - alpha / 2)),
    )

def save_diagnostic_plots(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    p_violence: np.ndarray,
    title_prefix: str,
    output_dir: Path,
):
    output_dir.mkdir(parents=True, exist_ok=True)

    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1],
    )

    fig, ax = plt.subplots(figsize=(5, 4))
    image = ax.imshow(cm)

    for (row, col), value in np.ndenumerate(cm):
        ax.text(col, row, str(value), ha="center", va="center")

    ax.set_xticks([0, 1], CLASS_NAMES)
    ax.set_yticks([0, 1], CLASS_NAMES)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.set_title(f"{title_prefix} — Confusion Matrix")
    fig.colorbar(image, ax=ax)
    fig.tight_layout()
    fig.savefig(
        output_dir / "confusion_matrix.png",
        dpi=180,
    )
    plt.close(fig)

    if len(np.unique(y_true)) == 2:
        fpr, tpr, _ = roc_curve(
            y_true,
            p_violence,
        )
        auc_value = roc_auc_score(
            y_true,
            p_violence,
        )

        fig, ax = plt.subplots(figsize=(5, 4))
        ax.plot(fpr, tpr, label=f"AUC={auc_value:.4f}")
        ax.plot([0, 1], [0, 1], linestyle="--")
        ax.set_xlabel("False Positive Rate")
        ax.set_ylabel("True Positive Rate")
        ax.set_title(f"{title_prefix} — ROC")
        ax.legend()
        ax.grid(True)
        fig.tight_layout()
        fig.savefig(
            output_dir / "roc_curve.png",
            dpi=180,
        )
        plt.close(fig)

        pr_precision, pr_recall, _ = precision_recall_curve(
            y_true,
            p_violence,
        )
        ap_value = average_precision_score(
            y_true,
            p_violence,
        )

        fig, ax = plt.subplots(figsize=(5, 4))
        ax.plot(
            pr_recall,
            pr_precision,
            label=f"AP={ap_value:.4f}",
        )
        ax.set_xlabel("Recall")
        ax.set_ylabel("Precision")
        ax.set_title(f"{title_prefix} — Precision-Recall")
        ax.legend()
        ax.grid(True)
        fig.tight_layout()
        fig.savefig(
            output_dir / "precision_recall_curve.png",
            dpi=180,
        )
        plt.close(fig)

## 7. Jalankan held-out evaluation

Cell ini hanya melakukan forward inference pada checkpoint yang sudah frozen.

In [12]:
results = []

ci_metric_names = [
    "accuracy",
    "f1_macro",
    "f1_violence",
    "roc_auc",
    "average_precision",
]

for dataset_position, (dataset_name, store) in enumerate(
    heldout_stores.items()
):
    print("\n" + "=" * 90)
    print(
        f"EVALUATE: {dataset_name} | checkpoint seed={RUN_SEED}"
    )
    print("=" * 90)

    dataset = HeldOutSingleFrameDataset(
        feature_path=store["feature_path"],
        label_path=store["label_path"],
        color_order=store["color_order"],
    )

    loader = make_loader(dataset)

    metrics, sample_indices, y_true, y_pred, probs = evaluate_loader(
        model,
        loader,
        DEVICE,
    )

    output_dir = EVAL_OUTPUT_DIR / dataset_name
    output_dir.mkdir(parents=True, exist_ok=True)

    source_paths = load_paths_if_available(
        store.get("paths_path"),
        expected_n=len(dataset),
    )

    prediction_df = pd.DataFrame(
        {
            "sample_index": sample_indices,
            "y_true": y_true,
            "y_pred": y_pred,
            "p_nonviolence": probs[:, 0],
            "p_violence": probs[:, 1],
        }
    )

    if source_paths is not None:
        prediction_df["video_path"] = [
            source_paths[int(index)]
            for index in sample_indices
        ]

    prediction_df.to_csv(
        output_dir / "predictions.csv",
        index=False,
    )

    for metric_position, metric_name in enumerate(ci_metric_names):
        ci_seed = (
            BOOTSTRAP_SEED
            + dataset_position * 100
            + metric_position
        )

        low, high = bootstrap_ci(
            y_true=y_true,
            y_pred=y_pred,
            p_violence=probs[:, 1],
            metric_name=metric_name,
            n_bootstrap=BOOTSTRAP_SAMPLES,
            seed=ci_seed,
        )

        metrics[f"{metric_name}_ci95_low"] = low
        metrics[f"{metric_name}_ci95_high"] = high

    metrics.update(
        {
            "variant": "single_frame",
            "checkpoint_seed": RUN_SEED,
            "checkpoint_path": str(CHECKPOINT_PATH),
            "best_epoch": checkpoint.get("epoch", np.nan),
            "best_val_f1_macro": checkpoint.get("best_score", np.nan),
            "dataset": dataset_name,
            "cache_source": store["cache_source"],
            "single_frame_index_0based": SINGLE_FRAME_INDEX,
            "single_frame_number_1based": SINGLE_FRAME_NUMBER_1BASED,
        }
    )

    results.append(metrics)

    save_diagnostic_plots(
        y_true=y_true,
        y_pred=y_pred,
        p_violence=probs[:, 1],
        title_prefix=f"SingleFrame seed {RUN_SEED} — {dataset_name}",
        output_dir=output_dir,
    )

    print(
        f"N={metrics['n']} | "
        f"AP={metrics['average_precision']:.4f} | "
        f"AUC={metrics['roc_auc']:.4f} | "
        f"Macro-F1={metrics['f1_macro']:.4f}"
    )

    del dataset, loader, probs

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

heldout_metrics_df = pd.DataFrame(results)

heldout_metrics_path = (
    EVAL_OUTPUT_DIR
    / f"heldout_metrics_single_frame_seed_{RUN_SEED}.csv"
)

heldout_metrics_df.to_csv(
    heldout_metrics_path,
    index=False,
)

display_columns = [
    "dataset",
    "n",
    "accuracy",
    "balanced_accuracy",
    "f1_macro",
    "precision_violence",
    "recall_violence",
    "f1_violence",
    "roc_auc",
    "average_precision",
]

print("\n=== HELD-OUT RESULTS ===")
display(
    heldout_metrics_df[display_columns].round(4)
)
print("Saved:", heldout_metrics_path)


EVALUATE: RWF | checkpoint seed=42


Inference:   0%|          | 0/125 [00:00<?, ?it/s]

N=1998 | AP=0.5839 | AUC=0.5912 | Macro-F1=0.5581

EVALUATE: UCF | checkpoint seed=42


Inference:   0%|          | 0/7 [00:00<?, ?it/s]

N=100 | AP=0.5735 | AUC=0.5054 | Macro-F1=0.4907

EVALUATE: BV | checkpoint seed=42


Inference:   0%|          | 0/88 [00:00<?, ?it/s]

N=1400 | AP=0.4888 | AUC=0.5192 | Macro-F1=0.5199

=== HELD-OUT RESULTS ===


,dataset,n,accuracy,balanced_accuracy,f1_macro,precision_violence,recall_violence,f1_violence,roc_auc,average_precision
0,RWF,1998,0.5696,0.5694,0.5581,0.6018,0.4088,0.4869,0.5912,0.5839
1,UCF,100,0.5200,0.5200,0.4907,0.5385,0.2800,0.3684,0.5054,0.5735
2,BV,1400,0.5236,0.5236,0.5199,0.5286,0.4357,0.4777,0.5192,0.4888


Saved: /content/drive/MyDrive/IVS/ViolenceDetection/Models/EffTempNet/HeldOut_Evaluation/SingleFrame/seed_42/heldout_metrics_single_frame_seed_42.csv


## 8. Manifest evaluasi

In [13]:
run_manifest = {
    "task": "heldout_evaluation_only",
    "training_performed": False,
    "variant": "single_frame",
    "checkpoint_seed": RUN_SEED,
    "checkpoint_path": str(CHECKPOINT_PATH),
    "checkpoint_epoch": checkpoint.get("epoch"),
    "checkpoint_best_val_f1_macro": checkpoint.get("best_score"),
    "device": str(DEVICE),
    "torch_version": torch.__version__,
    "timm_version": timm.__version__,
    "sequence_length_in_cache": SEQUENCE_LENGTH,
    "model_input_frames": MODEL_INPUT_FRAMES,
    "single_frame_index_0based": SINGLE_FRAME_INDEX,
    "single_frame_number_1based": SINGLE_FRAME_NUMBER_1BASED,
    "raw_resolution": [RAW_HEIGHT, RAW_WIDTH],
    "backbone_internal_resolution": [
        BACKBONE_INPUT_SIZE,
        BACKBONE_INPUT_SIZE,
    ],
    "normalization": "ImageNet",
    "use_amp": USE_AMP,
    "bootstrap_samples": BOOTSTRAP_SAMPLES,
    "bootstrap_seed": BOOTSTRAP_SEED,
    "heldout_datasets": list(HELDOUT_SPECS.keys()),
    "checkpoint_config": checkpoint_config,
}

manifest_path = (
    EVAL_OUTPUT_DIR / "evaluation_manifest.json"
)

with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(
        run_manifest,
        f,
        indent=2,
        default=str,
    )

print("Manifest saved:", manifest_path)
print("✅ EVALUATION-ONLY selesai.")
print("Tidak ada training atau backpropagation pada notebook ini.")

Manifest saved: /content/drive/MyDrive/IVS/ViolenceDetection/Models/EffTempNet/HeldOut_Evaluation/SingleFrame/seed_42/evaluation_manifest.json
✅ EVALUATION-ONLY selesai.
Tidak ada training atau backpropagation pada notebook ini.


## Untuk seed 0 dan 123

Dengan struktur folder training yang sama, **ubah hanya**:

```python
RUN_SEED = 0
```

atau:

```python
RUN_SEED = 123
```

Jangan mengubah `SEQUENCE_LENGTH`, `SINGLE_FRAME_INDEX`, preprocessing, atau cache held-out. Cache yang sama harus dipakai untuk seluruh checkpoint seed.